# Inpainting, Outpainting & Image Editing Notebook

> Hands-on Build It and Exercises.

## Build It

`code/main.py` implements a toy 1-D inpainting scheme on 5-dimensional data. We train a DDPM on 5-D mixture data where each sample is 5 floats from one of two clusters. At inference, we "mask" 2 of the 5 dimensions, inject the noisy-forward version of the unmasked three at each step, and regenerate only the masked dimensions.

### Step 1: 5-D DDPM data

In [ ]:
```python

def sample_data(rng):

    cluster = rng.choice([0, 1])

    center = [-1.0] * 5 if cluster == 0 else [1.0] * 5

    return [c + rng.gauss(0, 0.2) for c in center], cluster

In [ ]:
```

### Step 2: train denoiser over all 5 dims

Standard DDPM. Net outputs 5-D noise prediction for 5-D noisy input.

### Step 3: at inference, mask-aware reverse

In [ ]:
```python

def inpaint_step(x_t, mask, clean_image, alpha_bars, t, rng):

    # replace unmasked dims with a freshly noised version of the clean source

    a_bar = alpha_bars[t]

    for i in range(len(x_t)):

        if not mask[i]:

            x_t[i] = math.sqrt(a_bar) * clean_image[i] + math.sqrt(1 - a_bar) * rng.gauss(0, 1)

    # ...then run the normal reverse step on x_t

In [ ]:
```

This is the naive approach and it works on toy 1-D data. Real image inpainting uses the 9-channel input because texture coherence matters more.

### Step 4: outpainting

Outpainting is inpainting with the mask inverted: mask the new (previously non-existent) canvas, fill the rest with the original. Identical training objective.

## Exercises

In [ ]:
1. **Easy.** In `code/main.py`, vary the fraction of dimensions masked from 0.2 to 0.8. At what fraction does the inpaint quality (residual in masked dims) equal unconditional generation?
2. **Medium.** Implement RePaint: at every 10th reverse step, jump back 5 steps (add noise) and re-denoise. Measure whether it reduces boundary residual at the mask edge.
3. **Hard.** Use Hugging Face diffusers to compare: SD 1.5 Inpaint + ControlNet-Openpose vs Flux.1-Fill on 20 face-regeneration tasks. Score pose adherence and identity preservation separately.